# Team Member 2: Feature Extraction

This notebook prepares feature embeddings for the EPIC-KITCHENS action recognition project. It uses frozen pretrained encoders so the project remains feasible within three weeks.

## 1. Install packages
Run this cell in Colab/Kaggle.

In [ ]:
# Uncomment in Colab/Kaggle
# !pip install -q sentence-transformers open_clip_torch opencv-python torch torchvision pillow scikit-learn

## 2. Load metadata

In [ ]:
import os
import numpy as np
import pandas as pd

TRAIN_CSV = "P01_train.csv"
VAL_CSV = "P01_val.csv"
TEST_CSV = "P01_test.csv"
OUTPUT_DIR = "features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

for df in [train_df, val_df]:
    df["action_label"] = df["verb"].astype(str) + " " + df["noun"].astype(str)
    df["duration_frames"] = df["stop_frame"] - df["start_frame"] + 1

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
train_df.head()

## 3. Dataset check

In [ ]:
print("Top verbs:")
display(train_df["verb"].value_counts().head(10))
print("Top nouns:")
display(train_df["noun"].value_counts().head(10))
print("Top action labels:")
display(train_df["action_label"].value_counts().head(10))

## 4. Extract text features using Sentence-BERT
This converts each narration into a 384-dimensional vector.

In [ ]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_text_embeddings(df, split):
    texts = df["narration"].fillna("").astype(str).tolist()
    emb = text_model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    np.save(os.path.join(OUTPUT_DIR, f"{split}_text_embeddings.npy"), emb)
    print(split, emb.shape)
    return emb

train_text = get_text_embeddings(train_df, "train")
val_text = get_text_embeddings(val_df, "val")

## 5. Prepare object features
We use the noun label as a lightweight object cue. This satisfies the object-cue part without training an object detector.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

object_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_object = object_encoder.fit_transform(train_df[["noun_class"]].astype(str))
val_object = object_encoder.transform(val_df[["noun_class"]].astype(str))

np.save(os.path.join(OUTPUT_DIR, "train_object_features.npy"), train_object)
np.save(os.path.join(OUTPUT_DIR, "val_object_features.npy"), val_object)

print("Train object features:", train_object.shape)
print("Val object features:", val_object.shape)

## 6. Prepare labels for Team Member 3

In [ ]:
from sklearn.preprocessing import LabelEncoder

def save_label_files(label_col):
    le = LabelEncoder()
    train_labels = le.fit_transform(train_df[label_col].astype(str))
    val_labels = []
    known = set(le.classes_)
    for x in val_df[label_col].astype(str):
        val_labels.append(le.transform([x])[0] if x in known else -1)
    val_labels = np.array(val_labels)
    np.save(os.path.join(OUTPUT_DIR, f"train_{label_col}_labels.npy"), train_labels)
    np.save(os.path.join(OUTPUT_DIR, f"val_{label_col}_labels.npy"), val_labels)
    pd.DataFrame({"class_name": le.classes_, "class_id": range(len(le.classes_))}).to_csv(
        os.path.join(OUTPUT_DIR, f"{label_col}_label_mapping.csv"), index=False
    )
    print(label_col, len(le.classes_))

for col in ["verb_class", "noun_class", "action_label"]:
    save_label_files(col)

## 7. Visual feature extraction using CLIP
Run this section after Team Member 1 provides extracted video frames. Expected frame structure: `frames/{video_id}/frame_0000000001.jpg`.

In [ ]:
# Uncomment when image frames are available
# import torch
# from PIL import Image
# import open_clip
#
# FRAMES_ROOT = "frames"
# device = "cuda" if torch.cuda.is_available() else "cpu"
# clip_model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
# clip_model = clip_model.to(device).eval()
#
# def sample_frame_paths(row, num_frames=3):
#     start, stop = int(row["start_frame"]), int(row["stop_frame"])
#     frame_numbers = np.linspace(start, stop, num_frames).astype(int)
#     return [os.path.join(FRAMES_ROOT, row["video_id"], f"frame_{f:010d}.jpg") for f in frame_numbers]
#
# def get_visual_embeddings(df, split, num_frames=3):
#     features = []
#     missing = 0
#     with torch.no_grad():
#         for _, row in df.iterrows():
#             frame_feats = []
#             for path in sample_frame_paths(row, num_frames):
#                 if not os.path.exists(path):
#                     missing += 1
#                     continue
#                 img = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
#                 feat = clip_model.encode_image(img)
#                 feat = feat / feat.norm(dim=-1, keepdim=True)
#                 frame_feats.append(feat.cpu().numpy()[0])
#             features.append(np.mean(frame_feats, axis=0) if frame_feats else np.zeros(512))
#     features = np.vstack(features)
#     np.save(os.path.join(OUTPUT_DIR, f"{split}_visual_embeddings.npy"), features)
#     print(split, features.shape, "missing frames:", missing)
#     return features
#
# train_visual = get_visual_embeddings(train_df, "train", num_frames=3)
# val_visual = get_visual_embeddings(val_df, "val", num_frames=3)

## 8. Final handover checklist
After running the notebook, give Team Member 3 the `features/` folder containing text embeddings, object features, label files, and visual embeddings if frames are available.